#1. Bibliotecas


In [ ]:
# Importando as bibliotecas:
import pandas as pd          # para trabalhar com tabelas de dados
import numpy as np           # para cálculos numéricos
import matplotlib.pyplot as plt  # para criar gráficos
import seaborn as sns        # para gráficos mais bonitos

# Configurações visuais
plt.rcParams['figure.figsize'] = (12, 5)
sns.set_theme(style="whitegrid")

# Nova seção

#2. Carregando o dataset

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
df = pd.read_csv("/content/drive/MyDrive/CD/Acidentes_PorOcorrencia(25&26).csv")

print("Dataset carregado!")
print(f"Linhas: {df.shape[0]}")
print(f"Colunas: {df.shape[1]}")

FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/CD/Acidentes_PorOcorrencia(25&26).csv'

#3. Análise do estado dos dados

In [ ]:
df.head()   #Primeiras cinco linhas

In [ ]:
df.columns.tolist()   #Acessa o índice de nomes das colunas e o transforma em uma lista


In [ ]:
df.dtypes    #Mostra os tipos dos dados

In [ ]:
df.isnull().sum()   #Verificando a existência de dados faltantes

In [ ]:
duplicatas = df.duplicated().sum()    #Verificação de linhas duplicadas
print(f"Linhas duplicadas: {duplicatas}")

In [ ]:
df.describe()   #Estatísticas básicas das variáveis numéricas

In [ ]:
df.describe(include=object)   #Estatísticas básicas das variáveis categóricas

## Comentários:
- Data Inversa está como Object, mas deveria ser uma data
- Horário está como Object, mas deveria ser horário
- Latitude, Longitude e Km estão como float64
- No geral foram encontrados 81 dados faltantes, o que significa menos de 0,1% dos dados
- Não há linhas duplicadas
- Foram identificados valores extremos nas variáveis numéricas, detalhados nas seções de análise gráfica e outliers

# 4. Análises Gráficas


In [ ]:
# Acidentes por dia da semana
contagem = df['dia_semana'].value_counts()

plt.figure()
sns.barplot(x=contagem.index, y=contagem.values, palette='Blues_d')
plt.title('Acidentes por Dia da Semana')
plt.xlabel('Dia da Semana')
plt.ylabel('Quantidade de Acidentes')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

##Comentários:
  - Os dias com maior incidência de acidentes são sábado, domingo e sexta-feira, sugerindo correlação com deslocamentos de fim de semana e possível influência do consumo de álcool.

In [ ]:
# Top 10 causas de acidente
top_causas = df['causa_acidente'].value_counts().head(10)

plt.figure()
sns.barplot(x=top_causas.values, y=top_causas.index, palette='Reds_r')
plt.title('Top 10 Causas de Acidente')
plt.xlabel('Quantidade de Acidentes')
plt.ylabel('Causa')
plt.tight_layout()
plt.show()

## Comentários:
- A causa 'Ingestão de álcool' pode estar subestimada, pois casos não confirmados tendem a ser registrados como falhas de atenção ou reação. Os dados capturam o sintoma observável, não necessariamente a causa raiz.

In [ ]:
# Acidentes por fase do dia
fase = df['fase_dia'].value_counts()

plt.figure()
sns.barplot(x=fase.index, y=fase.values, palette='coolwarm')
plt.title('Acidentes por Fase do Dia')
plt.xlabel('Fase do Dia')
plt.ylabel('Quantidade de Acidentes')
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

In [ ]:
# Taxa de mortalidade por fase do dia
mortalidade = df.groupby('fase_dia')['mortos'].sum() / df.groupby('fase_dia')['mortos'].count()

plt.figure()
sns.barplot(x=mortalidade.index, y=mortalidade.values, palette='coolwarm')
plt.title('Taxa Média de Mortos por Acidente — Fase do Dia')
plt.xlabel('Fase do Dia')
plt.ylabel('Média de Mortos por Acidente')
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

## Comentários:
- Acidentes no período do amanhecer apresentam a maior taxa de mortalidade proporcional, possivelmente associados à fadiga acumulada de viagens noturnas e ao consumo de álcool.

In [ ]:
# Letalidade por hora (média de mortos por acidente)
df['hora'] = pd.to_datetime(df['horario'], format='%H:%M:%S', errors='coerce').dt.hour

letalidade = df.groupby('hora').agg(
    total_mortos=('mortos', 'sum'),
    total_acidentes=('mortos', 'count')
).reset_index()

letalidade['taxa_letalidade'] = letalidade['total_mortos'] / letalidade['total_acidentes'] * 100

plt.figure(figsize=(14, 5))

# Colorindo barras por intensidade
colors = ['#d73027' if v >= letalidade['taxa_letalidade'].quantile(0.75)
          else '#fc8d59' if v >= letalidade['taxa_letalidade'].median()
          else '#fee090'
          for v in letalidade['taxa_letalidade']]

bars = plt.bar(letalidade['hora'], letalidade['taxa_letalidade'], color=colors, edgecolor='white', width=0.8)

# Linha de média geral
media = letalidade['taxa_letalidade'].mean()
plt.axhline(media, color='#1a1a1a', linestyle='--', linewidth=1.2, label=f'Média geral: {media:.2f}%')

plt.title('Letalidade por Hora do Dia\n(% de mortos por acidente)', fontsize=13, fontweight='bold')
plt.xlabel('Hora do Dia', fontsize=11)
plt.ylabel('Taxa de Letalidade (%)', fontsize=11)
plt.xticks(range(0, 24), [f'{h:02d}h' for h in range(24)], rotation=45)
plt.legend()
plt.tight_layout()
plt.show()

# Imprimindo os horários mais letais
print("Top 5 horários mais letais:")
print(letalidade.nlargest(5, 'taxa_letalidade')[['hora','taxa_letalidade','total_mortos','total_acidentes']].to_string(index=False))

## Comentários:
- A madrugada (02h–05h) concentra a maior taxa de letalidade, não o maior volume de acidentes
- Isso indica que o risco de morte por acidente é determinado pelo horário, não pela frequência
- O pico de letalidade entre 02h–05h pode estar associado a fadiga, sono ao volante e menor presença de socorro
- A linha de média (≈0,X%) separa claramente os horários de alto risco dos demais


In [ ]:
# Top 10 estados com mais acidentes
top_uf = df['uf'].value_counts().head(10)

plt.figure()
sns.barplot(x=top_uf.index, y=top_uf.values, palette='viridis')
plt.title('Top 10 Estados com Mais Acidentes')
plt.xlabel('Estado')
plt.ylabel('Quantidade de Acidentes')
plt.tight_layout()
plt.show()

## Comentários:
- Os dados refletem apenas acidentes em rodovias federais. Estados com maior dependência de rodovias estaduais, como SP, podem estar subrepresentados.

In [ ]:
# Acidentes por condição meteorológica
clima = df['condicao_metereologica'].value_counts()

plt.figure()
sns.barplot(x=clima.values, y=clima.index, palette='Blues_r')
plt.title('Acidentes por Condição Meteorológica')
plt.xlabel('Quantidade de Acidentes')
plt.ylabel('Condição')
plt.tight_layout()
plt.show()

## Comentários:
- A predominância de acidentes em condições de céu claro indica que fatores climáticos não são determinantes, o comportamento humano, como distração e velocidade incompatível, responde pela maior parte das ocorrências.

In [ ]:
# Investigando os casos extremos
print("=== Acidente com mais veículos ===")
print(df[df['veiculos'] == df['veiculos'].max()][['data_inversa', 'uf', 'br', 'municipio', 'tipo_acidente', 'pessoas', 'mortos', 'veiculos']])

print("\n=== Acidente com mais mortos ===")
print(df[df['mortos'] == df['mortos'].max()][['data_inversa', 'uf', 'br', 'municipio', 'tipo_acidente', 'pessoas', 'mortos', 'veiculos']])

print("\n=== Acidente com mais ignorados ===")
print(df[df['ignorados'] == df['ignorados'].max()][['data_inversa', 'uf', 'br', 'municipio', 'tipo_acidente', 'pessoas', 'mortos', 'ignorados', 'veiculos']])

## Comentários:
- Foi identificado ao menos um registro com inconsistência grave (82 veículos, 81 ignorados, 2 pessoas) caracterizando erro de cadastro. Outliers legítimos, como acidentes com ônibus, devem ser preservados na análise.

In [ ]:
# Gravidade por tipo de acidente
gravidade_tipo = df.groupby('tipo_acidente')['mortos'].mean().sort_values(ascending=False).head(10)

plt.figure()
sns.barplot(x=gravidade_tipo.values, y=gravidade_tipo.index, palette='Reds_r')
plt.title('Média de Mortos por Tipo de Acidente (Top 10 mais letais)')
plt.xlabel('Média de Mortos por Acidente')
plt.ylabel('Tipo de Acidente')
plt.tight_layout()
plt.show()

## Comentários:
- Colisão frontal é o tipo de acidente mais letal (0,40 mortos/acidente), enquanto colisão traseira, o mais frequente, apresenta letalidade significativamente menor. Isso indica que políticas de segurança devem priorizar a prevenção de colisões frontais.

In [ ]:
# Filtrando apenas os traçados simples (sem combinações com ";")
tracado_simples = df[~df['tracado_via'].str.contains(';', na=False)]

let_tracado = tracado_simples.groupby('tracado_via')['mortos'].mean().sort_values(ascending=False)

plt.figure()
sns.barplot(x=let_tracado.values, y=let_tracado.index, palette='Oranges_r')
plt.title('Média de Mortos por Acidente — Traçado da Via (Simples)')
plt.xlabel('Média de Mortos por Acidente')
plt.ylabel('Traçado da Via')
plt.tight_layout()
plt.show()

print(f"\nRegistros com traçado simples: {len(tracado_simples)} de {len(df)}")

## Comentários:
- Túneis e curvas apresentam a maior letalidade por acidente, enquanto rotatórias têm a menor, sugerindo que elementos que forçam redução de velocidade são protetores.

In [ ]:
# Série temporal correta — agrupando por ano e mês
df['ano_mes'] = df['data_inversa'].dt.to_period('M')
acidentes_periodo = df.groupby('ano_mes').size()

plt.figure(figsize=(14, 5))
sns.lineplot(x=acidentes_periodo.index.astype(str), y=acidentes_periodo.values, marker='o', color='steelblue')
plt.title('Acidentes por Mês — Série Temporal Completa')
plt.xlabel('Mês/Ano')
plt.ylabel('Quantidade de Acidentes')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## Comentários:
- A tabela revela pico de acidentes em Dezembro/2025, consistente com o período de festas e viagens de fim de ano. Os meses de Jan-Fev/2026 apresentam queda abrupta que pode refletir dados incompletos, e não redução real de acidentes.

In [ ]:
# Matriz de correlação
numericas = ['pessoas', 'mortos', 'feridos_leves', 'feridos_graves', 'ilesos', 'ignorados', 'feridos', 'veiculos']

plt.figure(figsize=(10, 8))
sns.heatmap(df[numericas].corr(), annot=True, fmt='.2f', cmap='coolwarm', center=0)
plt.title('Matriz de Correlação — Variáveis Numéricas')
plt.tight_layout()
plt.show()

## Comentários:
- Mortos apresenta correlação muito baixa com todas as outras variáveis numéricas, indicando que a letalidade de um acidente é determinada pelo tipo e condições do acidente, não pelo volume de pessoas ou veículos envolvidos.

# 5. Separação de Variáveis

In [ ]:
# Definindo a variável alvo
target = 'classificacao_acidente'

# Separando variáveis numéricas (excluindo id e target)
vars_numericas = df.select_dtypes(include=['int64', 'float64']).columns.tolist()
vars_numericas = [col for col in vars_numericas if col != 'id']

# Separando variáveis categóricas (excluindo target e colunas administrativas)
vars_categoricas = df.select_dtypes(include=['object']).columns.tolist()
vars_categoricas = [col for col in vars_categoricas
                    if col not in [target, 'data_inversa', 'horario',
                                   'municipio', 'regional', 'delegacia', 'uop']]

print(f"Variável alvo: {target}")
print(f"\nVariáveis numéricas ({len(vars_numericas)}):")
print(vars_numericas)
print(f"\nVariáveis categóricas relevantes ({len(vars_categoricas)}):")
print(vars_categoricas)

## Comentários:
- A variável alvo definida é `classificacao_acidente`, que categoriza cada acidente em Fatal, Com Feridos ou Sem Vítimas
- Foram identificadas 12 variáveis numéricas e 10 categóricas relevantes para o modelo
- Colunas administrativas como `municipio`, `delegacia` e `uop` foram excluídas por serem muito fragmentadas (alta cardinalidade) e pouco generalizáveis para o modelo
- `id` foi excluído por ser apenas um identificador, sem valor preditivo
- `feridos` é a soma de `feridos_leves` + `feridos_graves`, o que indica possível redundância — será investigada no teste de correlação de Pearson
- `km`, `latitude` e `longitude` constam como numéricas, mas foram identificadas anteriormente com problema de vírgula decimal — serão tratadas na etapa de pré-processamento

#6. Análise da Variável Alvo

In [ ]:
# Distribuição da variável alvo
alvo = df[target].value_counts()
alvo_pct = (alvo / alvo.sum() * 100).round(2)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Gráfico de barras
sns.barplot(x=alvo.values, y=alvo.index, palette='RdYlGn_r', ax=axes[0])
axes[0].set_title('Distribuição da Variável Alvo')
axes[0].set_xlabel('Quantidade')
axes[0].set_ylabel('Classe')

# Gráfico de pizza
axes[1].pie(alvo.values, labels=alvo.index, autopct='%1.1f%%',
            colors=['#d73027','#fee08b','#1a9850'], startangle=90)
axes[1].set_title('Proporção das Classes')

plt.tight_layout()
plt.show()

print("Distribuição absoluta e percentual:")
for classe, qtd, pct in zip(alvo.index, alvo.values, alvo_pct.values):
    print(f"  {classe}: {qtd} registros ({pct}%)")

## Comentários:
- O dataset é fortemente desbalanceado: 77,5% dos registros pertencem à classe "Com Vítimas Feridas"
- "Com Vítimas Fatais" representa apenas 7,1% dos dados — a classe minoritária
- Esse desbalanceamento é um problema crítico para o kNN, pois o algoritmo tende a favorecer
  a classe majoritária nas previsões, comprometendo a detecção dos casos fatais
- Na aplicação prática, errar na identificação de um acidente fatal é o pior tipo de erro possível
  (falso negativo), pois pode resultar no envio de recursos insuficientes ao local
- Estratégias de balanceamento (como SMOTE ou undersampling) serão necessárias antes de treinar o modelo

# 7.1 Análise Univariada

In [ ]:
# Distribuição das variáveis categóricas relevantes
for col in vars_categoricas:
    contagem = df[col].value_counts().head(8)

    plt.figure(figsize=(12, 4))
    sns.barplot(x=contagem.values, y=contagem.index, palette='Blues_r')
    plt.title(f'Distribuição: {col}')
    plt.xlabel('Quantidade de Acidentes')
    plt.ylabel(col)
    plt.tight_layout()
    plt.show()

    print(f"\nTop valores — {col}:")
    print(contagem.to_string())
    print("-" * 50)

#7.2 .Análise Bivariada

In [ ]:
# Relação entre variável alvo e variáveis categóricas
for col in ['fase_dia', 'tipo_acidente', 'condicao_metereologica', 'tipo_pista', 'uso_solo']:
    ct = pd.crosstab(df[col], df[target], normalize='index') * 100

    ct.plot(kind='bar', figsize=(12, 4), colormap='RdYlGn_r', edgecolor='white')
    plt.title(f'Proporção da Classificação do Acidente por {col}')
    plt.xlabel(col)
    plt.ylabel('Proporção (%)')
    plt.xticks(rotation=30)
    plt.legend(title='Classificação', bbox_to_anchor=(1.05, 1))
    plt.tight_layout()
    plt.show()

## Comentários:

**fase_dia:**
- Amanhecer tem a maior proporção de vítimas fatais (~14%), confirmando a análise de mortalidade anterior
- Pleno dia tem a menor proporção de fatais (~6%), apesar do maior volume absoluto
- A proporção de "Sem Vítimas" é relativamente estável entre os períodos

**tipo_acidente:**
- Colisão frontal e Atropelamento de Pedestre se destacam com alta proporção de vítimas fatais (~28-30%)
- Incêndio apresenta ~100% de "Sem Vítimas" — poucos registros, mas padrão interessante
- Colisão traseira, o tipo mais frequente, tem proporção de fatais muito baixa (~3%)
- Derramamento de carga tem ~47% "Sem Vítimas" — faz sentido, é um evento material
- Esses padrões reforçam que o tipo de acidente é uma feature discriminativa forte para o modelo

**condicao_metereologica:**
- Neve apresenta ~100% "Com Vítimas Feridas" — mas são poucos registros, resultado pouco confiável
- As demais condições têm distribuições muito similares entre si
- "Ignorado" tem proporção ligeiramente maior de "Sem Vítimas" (~20%) — possivelmente registros de menor gravidade onde o clima não foi anotado
- Condição meteorológica parece ter baixo poder discriminativo para o target

**tipo_pista:**
- Pista simples tem proporção de fatais maior (~10%) vs dupla (~5%) e múltipla (~4%)
- Faz sentido: pista simples não tem separação física entre sentidos, favorecendo colisões frontais
- Pista múltipla tem a menor letalidade proporcional

**uso_solo:**
- Área rural ("Não") tem proporção de fatais quase o dobro da urbana (~8% vs ~4%)
- Faz sentido: rodovias rurais têm maior velocidade, menor iluminação e socorro mais distante
- Diferença relevante — uso_solo deve ser uma feature importante para o modelo

#8. Missingness

In [ ]:
import missingno as msno

# Substituindo valores ausentes disfarçados por NaN real
df_miss = df.copy()
df_miss['condicao_metereologica'] = df_miss['condicao_metereologica'].replace('Ignorado', np.nan)
df_miss['sentido_via'] = df_miss['sentido_via'].replace('Não Informado', np.nan)

# Mapa visual de ausências
plt.figure(figsize=(14, 6))
msno.matrix(df_miss, figsize=(14, 6), fontsize=10, color=(0.2, 0.4, 0.8))
plt.title('Mapa de Valores Ausentes')
plt.tight_layout()
plt.show()

# Contagem total de ausentes por coluna
ausentes = df_miss.isnull().sum()
ausentes = ausentes[ausentes > 0].sort_values(ascending=False)
pct = (ausentes / len(df_miss) * 100).round(3)

print("Valores ausentes por coluna:")
for col in ausentes.index:
    print(f"  {col}: {ausentes[col]} ({pct[col]}%)")

In [ ]:
# Classificação do tipo de missingness por coluna

# Verificando se ausências em condicao_metereologica se concentram em algum período
print("=== condicao_metereologica — ausentes por fase_dia ===")
print(df_miss[df_miss['condicao_metereologica'].isnull()]['fase_dia'].value_counts())

print("\n=== condicao_metereologica — ausentes por uf ===")
print(df_miss[df_miss['condicao_metereologica'].isnull()]['uf'].value_counts().head(5))

print("\n=== sentido_via — ausentes por tipo_pista ===")
print(df_miss[df_miss['sentido_via'].isnull()]['tipo_pista'].value_counts())

print("\n=== classificacao_acidente — registros ausentes ===")
print(df_miss[df_miss['classificacao_acidente'].isnull()][['data_inversa','uf','tipo_acidente','fase_dia']])

In [ ]:
prop_ignorado = df.groupby('tipo_acidente')['condicao_metereologica'].apply(
    lambda x: (x == 'Ignorado').sum() / len(x) * 100
).sort_values(ascending=False).head(10)

plt.figure(figsize=(12, 5))
sns.barplot(x=prop_ignorado.values, y=prop_ignorado.index, palette='Reds_r')
plt.title('% de "Ignorado" em condicao_metereologica por Tipo de Acidente')
plt.xlabel('% de registros com valor Ignorado')
plt.tight_layout()
plt.show()

## Comentários:
- Atropelamento de Animal e Saída de Pista apresentam as maiores proporções de "Ignorado" em condicao_metereologica
- Isso sugere que o valor "Ignorado" não é aleatório — certos tipos de acidente têm sistematicamente menos registro das condições climáticas
- Esse padrão é consistente com a classificação MAR (Missing At Random condicionado ao tipo de acidente)

## Comentários — Classificação do Tipo de Missingness:

**condicao_metereologica — MAR (Missing At Random)**
- 67% das ausências ocorrem em Plena Noite (746 de 1114)
- A ausência depende de outra variável observada (fase_dia)
- Explicação: à noite é mais difícil observar e registrar a condição do tempo no local do acidente
- Estratégia: imputação pela moda dentro de cada grupo de fase_dia

**sentido_via — MCAR (Missing Completely At Random)**
- Ausências distribuídas proporcionalmente entre tipos de pista
- Sem padrão identificável — ausência parece aleatória
- Estratégia: imputação pela moda geral ou remoção das linhas

**delegacia / uop / regional — MCAR (Missing Completely At Random)**
- Pouquíssimos registros ausentes (<0.06%)
- Colunas administrativas que serão excluídas do modelo de qualquer forma
- Estratégia: preencher com 'Não Informado' ou remover linhas

**classificacao_acidente — MNAR (Missing Not At Random)**
- Apenas 2 registros ausentes, ambos acidentes graves (Colisão frontal e Tombamento)
- A ausência pode estar relacionada à gravidade: registros muito graves podem ter ficado incompletos no sistema
- Estratégia: remover as 2 linhas — impacto desprezível no dataset

#9. Detecção de Duplicatas Fuzzy

In [ ]:
# Detecção de duplicatas fuzzy por similaridade de campos-chave
# Estratégia: agrupar por UF + BR + data e verificar registros
# com horário muito próximo (possível duplo registro do mesmo acidente)

df['data_inversa'] = pd.to_datetime(df['data_inversa'])
df['horario'] = pd.to_datetime(df['horario'], format='%H:%M:%S', errors='coerce').dt.time

# Agrupando por UF, BR e data
grupos = df.groupby(['uf', 'br', 'data_inversa'])

suspeitos = []

for nome, grupo in grupos:
    if len(grupo) < 2:
        continue
    grupo_sorted = grupo.sort_values('horario').reset_index()
    for i in range(len(grupo_sorted) - 1):
        h1 = grupo_sorted.loc[i, 'horario']
        h2 = grupo_sorted.loc[i+1, 'horario']
        if h1 is not None and h2 is not None:
            from datetime import datetime, timedelta
            t1 = datetime.combine(datetime.today(), h1)
            t2 = datetime.combine(datetime.today(), h2)
            diff = abs((t2 - t1).total_seconds() / 60)
            if diff <= 5:
                suspeitos.append(grupo_sorted.loc[i, 'index'])
                suspeitos.append(grupo_sorted.loc[i+1, 'index'])

suspeitos = list(set(suspeitos))
print(f"Registros suspeitos de duplicata fuzzy: {len(suspeitos)}")

if len(suspeitos) > 0:
    print("\nAmostra dos casos suspeitos:")
    print(df.loc[suspeitos[:10], ['data_inversa', 'uf', 'br', 'km', 'horario',
                                   'tipo_acidente', 'causa_acidente', 'pessoas']])

In [ ]:
# Refinando a busca — exigindo também km idêntico ou muito próximo
suspeitos_refinados = []

for nome, grupo in grupos:
    if len(grupo) < 2:
        continue
    grupo_sorted = grupo.sort_values('horario').reset_index()
    for i in range(len(grupo_sorted) - 1):
        h1 = grupo_sorted.loc[i, 'horario']
        h2 = grupo_sorted.loc[i+1, 'horario']
        km1 = grupo_sorted.loc[i, 'km']
        km2 = grupo_sorted.loc[i+1, 'km']
        if h1 is not None and h2 is not None:
            from datetime import datetime
            t1 = datetime.combine(datetime.today(), h1)
            t2 = datetime.combine(datetime.today(), h2)
            diff_tempo = abs((t2 - t1).total_seconds() / 60)
            try:
                diff_km = abs(float(str(km1).replace(',','.')) - float(str(km2).replace(',','.')))
            except:
                diff_km = 999
            if diff_tempo <= 2 and diff_km <= 1:
                suspeitos_refinados.append(grupo_sorted.loc[i, 'index'])
                suspeitos_refinados.append(grupo_sorted.loc[i+1, 'index'])

suspeitos_refinados = list(set(suspeitos_refinados))
print(f"Registros suspeitos (critério refinado): {len(suspeitos_refinados)}")

if len(suspeitos_refinados) > 0:
    print("\nAmostra:")
    print(df.loc[suspeitos_refinados[:10], ['data_inversa', 'uf', 'br', 'km',
                                             'horario', 'tipo_acidente', 'pessoas', 'mortos']])

## Comentários:
- A busca inicial com critério de 5 minutos retornou 3.524 suspeitos — amplo demais,
  capturando acidentes distintos em trechos movimentados da mesma rodovia
- Com critério refinado (2 minutos + km com diferença ≤ 1) foram encontrados 179 registros suspeitos
- O caso mais evidente é o par de linhas 33815/33817: mesmo local, mesmo tipo de acidente,
  2 minutos de diferença — forte indicativo de duplo registro do mesmo evento
- Os demais casos exigem análise manual para confirmação
- Estratégia: marcar os 179 registros para revisão manual antes do treinamento do modelo;
  remover apenas os pares com km idêntico e tipo de acidente igual
- Impacto máximo: 179 registros de 83.909 (0,21%) — baixo, mas importante corrigir para não introduzir viés no modelo


# 10. Testes Estatísticos

In [ ]:
from scipy import stats

# Correlação de Pearson entre variáveis numéricas e mortos
print("=== Correlação de Pearson com 'mortos' ===\n")

for col in vars_numericas:
    if col == 'mortos':
        continue
    # Removendo NaN para o cálculo
    dados = df[['mortos', col]].dropna()
    r, p = stats.pearsonr(dados['mortos'], dados[col])
    significativo = "✓ significativo" if p < 0.05 else "✗ não significativo"
    print(f"{col:20s} | r = {r:+.4f} | p = {p:.4f} | {significativo}")

In [ ]:
# Teste Qui-quadrado entre variáveis categóricas e a variável alvo
print("=== Teste Qui-quadrado com 'classificacao_acidente' ===\n")

for col in vars_categoricas:
    tabela = pd.crosstab(df[col], df[target])
    chi2, p, dof, _ = stats.chi2_contingency(tabela)
    significativo = "✓ significativo" if p < 0.05 else "✗ não significativo"
    print(f"{col:30s} | chi2 = {chi2:10.2f} | p = {p:.4f} | gl = {dof:3d} | {significativo}")

## Comentários:

**Correlação de Pearson:**
- Dataset grande (83.909 registros) faz com que correlações mínimas apareçam como
  estatisticamente significativas — o valor de r é mais importante que o p-valor aqui
- `pessoas` tem a correlação mais relevante com mortos (r = +0.18): mais pessoas envolvidas,
  ligeiramente mais mortos
- `feridos_leves` tem correlação negativa (r = -0.07): acidentes com muitos feridos leves
  tendem a ter menos mortos — perfis de acidente distintos
- `ilesos` e `feridos` são os únicos genuinamente não significativos (p > 0.05)
- `br` e `km` têm r ≈ 0.02 — correlação praticamente nula apesar do p significativo;
  o número da rodovia e o km isoladamente não explicam mortalidade
- Nenhuma variável numérica tem correlação forte com mortos, confirmando que
  a letalidade depende de fatores qualitativos (tipo, traçado, fase do dia)

**Teste Qui-quadrado:**
- Todas as variáveis categóricas têm associação estatisticamente significativa com
  a variável alvo (p = 0.0000 em todas)
- `tipo_acidente` tem o maior chi2 (21.035) — é a variável com maior associação
  com a classificação do acidente, confirmando o que vimos na análise bivariada
- `causa_acidente` vem em segundo (chi2 = 14.128) — também forte discriminador
- `condicao_metereologica` tem o menor chi2 (152) entre as categóricas — menor
  poder discriminativo, consistente com a análise bivariada que mostrou distribuições
  similares entre condições climáticas
- `tracado_via` tem 1.284 graus de liberdade — reflexo das 643 combinações únicas
  identificadas anteriormente; alta fragmentação reduz a confiabilidade do teste
- Conclusão: todas as variáveis categóricas selecionadas são relevantes para o modelo,
  com destaque para tipo_acidente e causa_acidente

#11. Pairplot

In [ ]:
# Verificar histogramas gerais
colunas = ['pessoas', 'mortos', 'feridos_leves', 'feridos_graves', 'ilesos',
           'veiculos', 'br', 'km', 'ignorados', 'feridos']

fig, axes = plt.subplots(5, 2, figsize=(12, 4 * 5))
axes = axes.flatten()

for i, col in enumerate(colunas):
    sns.histplot(df[col], bins=30, ax=axes[i], color='steelblue', kde=True)
    axes[i].set_title(f'Distribuição: {col}')
    axes[i].set_xlabel(col)
    axes[i].set_ylabel('Frequência')

plt.tight_layout()
plt.show()

In [ ]:
# Pairplot das variáveis numéricas mais relevantes colorido pelo target
vars_pairplot = ['pessoas', 'mortos', 'feridos_graves', 'veiculos', 'ignorados']

sns.pairplot(
    df[vars_pairplot + [target]].dropna(),
    hue=target,
    palette={'Com Vítimas Fatais': '#d73027',
             'Com Vítimas Feridas': '#fee08b',
             'Sem Vítimas': '#1a9850'},
    plot_kws={'alpha': 0.3, 's': 15},
    diag_kind='kde'
)
plt.suptitle('Pairplot — Variáveis Numéricas por Classificação do Acidente', y=1.02)
plt.tight_layout()
plt.show()

## Comentários:
- A diagonal (KDE) confirma a distribuição assimétrica à direita em todas as variáveis:
  alta concentração próxima de zero com cauda longa - como visto nos histogramas
- As três classes se sobrepõem bastante em quase todos os pares, indicando que
  nenhuma variável numérica isoladamente separa bem as classes
- `mortos` é a única variável que mostra separação visual clara:
  pontos vermelhos (Com Vítimas Fatais) aparecem exclusivamente acima de zero,
  enquanto amarelos e verdes se concentram em zero
- O par `veiculos x ignorados` mostra correlação positiva visível (já identificada
  na matriz de correlação com r = 0.73) — quanto mais veículos, mais ignorados
- `feridos_graves x mortos` mostra que acidentes fatais (vermelho) tendem a ter
  mais feridos graves simultaneamente — padrão esperado em acidentes severos
- A sobreposição das classes nas variáveis numéricas reforça a importância das
  variáveis categóricas (tipo_acidente, causa_acidente) para discriminar o target,
  conforme evidenciado pelo teste Qui-quadrado
- O outlier de 82 veículos aparece visualmente isolado nos gráficos com `veiculos`,
  reforçando a necessidade de remoção desse registro antes do treinamento

#12. Ruídos e Outliers


In [ ]:
# Distribuição das variáveis numéricas principais (Histogramas)
fig, axes = plt.subplots(2, 3, figsize=(16, 10))

colunas = ['pessoas', 'mortos', 'feridos_leves', 'feridos_graves', 'ilesos', 'veiculos']

for i, col in enumerate(colunas):

  ax = axes[i // 3][i % 3]
  sns.histplot(df[col], bins=30, ax=ax, color='steelblue')
  ax.set_title(f'Distribuição: {col}')
  ax.set_xlabel(col)
  ax.set_ylabel('Frequência')

plt.tight_layout()
plt.show()

## Comentários:
- Todas as variáveis numéricas apresentam distribuição assimétrica à direita, com alta concentração de valores baixos e outliers significativos. A mediana é mais representativa que a média para descrever o perfil típico de acidente.
- O boxplot a seguir identifica formalmente esses outliers

In [ ]:
# Boxplot das variáveis numéricas principais
fig, axes = plt.subplots(1, 4, figsize=(16, 6))

colunas_box = ['pessoas', 'mortos', 'feridos', 'veiculos']

for i, col in enumerate(colunas_box):
    sns.boxplot(y=df[col], ax=axes[i], color='steelblue')
    axes[i].set_title(col)

plt.suptitle('Distribuição e Outliers — Variáveis Numéricas', y=1.02)
plt.tight_layout()
plt.show()

## Comentários:
- O boxplot confirma a distribuição assimétrica identificada nos histogramas.
  A caixa azul (que representa 50% central dos dados) está achatada próxima de zero em todas as variáveis, indicando que a maioria dos acidentes envolve poucos elementos (1-2 pessoas, 0 mortos, 1-2 veículos).
  Os círculos acima da linha superior são outliers formalmente identificados pelo método IQR (Intervalo Interquartil):
- 'veiculos': o ponto isolado em 82 corresponde ao registro corrompido identificado anteriormente (linha 10260), e aparece graficamente separado
de todos os demais — reforçando que se trata de erro de cadastro.
- 'mortos': outliers graduais e contínuos entre 2 e 16, sugerindo eventos reais de gravidades variadas, não erros de digitação.
- 'feridos': maior concentração de outliers, consistente com acidentes envolvendo ônibus ou engavetamentos com muitas vítimas.
- Conclusão: a mediana é a medida mais adequada para descrever o perfil típico
  de acidente neste dataset, pois a média é distorcida pelos outliers

In [ ]:
# Detecção formal de outliers pelo método IQR
print("=== Análise de Outliers — Método IQR ===\n")

vars_outlier = ['pessoas', 'mortos', 'feridos_leves', 'feridos_graves',
                'ilesos', 'ignorados', 'feridos', 'veiculos']

resumo_outliers = []

for col in vars_outlier:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    limite_inf = Q1 - 1.5 * IQR
    limite_sup = Q3 + 1.5 * IQR

    outliers = df[(df[col] < limite_inf) | (df[col] > limite_sup)]
    pct = round(len(outliers) / len(df) * 100, 2)  # corrigido aqui

    resumo_outliers.append({
        'variavel': col,
        'Q1': Q1, 'Q3': Q3, 'IQR': IQR,
        'limite_inf': limite_inf,
        'limite_sup': limite_sup,
        'n_outliers': len(outliers),
        'pct_outliers': pct
    })

    print(f"{col:20s} | Q1={Q1:.1f} Q3={Q3:.1f} IQR={IQR:.1f} "
          f"| limite_sup={limite_sup:.1f} | outliers={len(outliers)} ({pct}%)")

resumo_df = pd.DataFrame(resumo_outliers)

In [ ]:
# Complemento: Z-score para variáveis com IQR = 0
from scipy.stats import zscore

print("=== Análise de Outliers — Método Z-score (threshold = 3) ===\n")

for col in vars_outlier:
    z = zscore(df[col].dropna())
    outliers_z = (abs(z) > 3).sum()
    pct_z = round(outliers_z / len(df) * 100, 2)
    print(f"{col:20s} | outliers Z-score: {outliers_z} ({pct_z}%)")

## Comentários:

**Método IQR:**
- `mortos`, `feridos_graves` e `feridos` apresentaram IQR = 0, pois mais de 75%
  dos valores são zero ou um — nesse caso o método IQR classifica qualquer valor
  diferente como outlier, gerando resultados distorcidos (até 42,88% em `feridos`)
- O método IQR não é adequado para variáveis com distribuição altamente concentrada
  num único valor, como é o caso das variáveis de vítimas neste dataset

**Método Z-score (mais adequado para este dataset):**
- Os percentuais ficam entre 0,64% e 2,7% — muito mais realistas
- `ignorados` tem a maior proporção de outliers (2,7%) nos dois métodos —
  consistente com o registro corrompido da linha 10260 (81 ignorados)
- `veiculos` e `feridos` também apresentam outliers relevantes (~1,3% e ~1,24%)
- Os demais ficam abaixo de 1% — percentual baixo considerando o volume do dataset

**Conclusão sobre tratamento:**
- Outliers por erro de cadastro (linha 10260): remover antes do modelo
- Outliers legítimos (acidentes graves com ônibus, engavetamentos): manter,
  pois representam eventos reais importantes para o aprendizado do modelo
- Não há necessidade de suavização — os outliers legítimos são eventos discretos
  e reais, não ruído de medição contínua
- Estratégia recomendada: remoção cirúrgica apenas dos registros comprovadamente
  incorretos, preservando a distribuição natural dos dados

In [ ]:
!jupyter nbconvert --to latex /content/CD_Equipe4.ipynb

In [ ]:
!zip -r projeto_latex.zip . -i "CD_Equipe4.tex" "CD_Equipe4_files/*"

# Entrega 2 — Limpeza, Normalização, Encoding e kNN

Nesta seção, damos continuidade à Entrega 1, utilizando o diagnóstico da Análise Exploratória de Dados como base para as etapas iniciais de pré-processamento.

O objetivo desta etapa é tratar valores ausentes, corrigir inconsistências, aplicar encoding em variáveis categóricas, normalizar atributos numéricos e avaliar o impacto dessas transformações por meio de um modelo kNN.

## 1. Diagnóstico herdado da Entrega 1

A Entrega 1 identificou problemas importantes para a etapa de modelagem, como valores ausentes explícitos e valores ausentes disfarçados, inconsistências em registros específicos, variáveis redundantes, alta cardinalidade em algumas colunas categóricas, fragmentação da variável `tracado_via` e desbalanceamento da variável-alvo `classificacao_acidente`.

Com base nesse diagnóstico, a Entrega 2 aplica um pipeline de pré-processamento para preparar os dados para o treinamento e avaliação de um modelo kNN.

In [2]:
import pandas as pd
import numpy as np

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
import os

os.listdir("/content/drive/MyDrive")

['UFPE Declaração .pdf',
 'Classroom',
 'IMG_9068.png',
 'IMG_0408.png',
 'IMG_0413.png',
 'Atestado de Matrícula - Eduardo Neves.pdf',
 'Comprovante_29-10-2025_125130.pdf',
 'Colab Notebooks',
 'Cópia de risco_cardiovascular_base.csv',
 'Lista02.pdf',
 'ClinicaNutricao.gdoc',
 'EDOO .gdoc',
 'Documento sem título (2).gdoc',
 'Documento sem título (1).gdoc',
 'strateegia ds.gdoc',
 'datatran2025.csv',
 'Planilha sem título.gsheet',
 'Lista 1 - AVLC 2026.1 (2).gdoc',
 'arquivos_tran202526.csv',
 'Documento sem título.gdoc',
 'Relatorio_EDA_PRF_Eduardo.gdoc',
 'cell challenge.gdoc',
 'requerimento-atualizacao-registro-PJ-CREF12.gdoc',
 'EDOO - Relatório do projeto de POO.gdoc',
 'Entrega - Documentação da Arquitetura do Projeto',
 'Entrega - Documentação da Arquitetura do Projeto (1).gdoc',
 'Entrega - Documentação da Arquitetura do Projeto.gdoc',
 'entrega 30 04 - documentação da arquitetura.gdoc',
 'Entrega 30 04 - documentação da arquitetura.gdoc',
 'Entrega 30 04 -

## 2. Carregamento do dataset

In [5]:
df = pd.read_csv(
    "/content/drive/MyDrive/CD/arquivos_tran202526.csv",
    encoding="utf-8"
)

df.head()

,id,data_inversa,dia_semana,horario,uf,br,km,municipio,causa_acidente,tipo_acidente,...,feridos_graves,ilesos,ignorados,feridos,veiculos,latitude,longitude,regional,delegacia,uop
0,652493,2025-01-01,quarta-feira,06:20:00,SP,116,225,GUARULHOS,Reação tardia ou ineficiente do condutor,Tombamento,...,0,0,1,1,2,"-23,48586772","-46,54075317",SPRF-SP,DEL01-SP,UOP01-DEL01-SP
1,652519,2025-01-01,quarta-feira,07:50:00,CE,116,"546,2",PENAFORTE,Pista esburacada,Colisão frontal,...,0,1,4,1,6,"-7,812288","-39,08333306",SPRF-CE,DEL05-CE,UOP03-DEL05-CE
2,652522,2025-01-01,quarta-feira,08:45:00,PR,369,"88,2",CORNELIO PROCOPIO,Reação tardia ou ineficiente do condutor,Colisão traseira,...,0,2,0,3,2,"-23,182565","-50,637228",SPRF-PR,DEL07-PR,UOP05-DEL07-PR
3,652544,2025-01-01,quarta-feira,11:00:00,PR,116,74,CAMPINA GRANDE DO SUL,Reação tardia ou ineficiente do condutor,Saída de leito carroçável,...,0,4,0,1,2,"-25,36517687","-49,04223028",SPRF-PR,DEL01-PR,UOP02-DEL01-PR
4,652549,2025-01-01,quarta-feira,09:30:00,MG,251,471,FRANCISCO SA,Velocidade Incompatível,Colisão frontal,...,1,1,2,2,4,"-16,46801304","-43,43121303",SPRF-MG,DEL12-MG,UOP01-DEL12-MG


In [6]:
print("Dataset carregado!")
print(f"Linhas: {df.shape[0]}")
print(f"Colunas: {df.shape[1]}")

Dataset carregado!
Linhas: 83909
Colunas: 30


## 3. Verificação inicial após carregamento

Nesta etapa, verificamos a estrutura do dataset carregado, os tipos de dados das colunas, a presença de valores ausentes explícitos e a distribuição da variável-alvo `classificacao_acidente`.

In [7]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 83909 entries, 0 to 83908
Data columns (total 30 columns):
 #   Column                  Non-Null Count  Dtype 
---  ------                  --------------  ----- 
 0   id                      83909 non-null  int64 
 1   data_inversa            83909 non-null  object
 2   dia_semana              83909 non-null  object
 3   horario                 83909 non-null  object
 4   uf                      83909 non-null  object
 5   br                      83909 non-null  int64 
 6   km                      83909 non-null  object
 7   municipio               83909 non-null  object
 8   causa_acidente          83909 non-null  object
 9   tipo_acidente           83909 non-null  object
 10  classificacao_acidente  83907 non-null  object
 11  fase_dia                83909 non-null  object
 12  sentido_via             83909 non-null  object
 13  condicao_metereologica  83909 non-null  object
 14  tipo_pista              83909 non-null  object
 15  tr

In [8]:
df.isnull().sum().sort_values(ascending=False)

,0
uop,48
delegacia,27
regional,4
classificacao_acidente,2
dia_semana,0
id,0
br,0
km,0
causa_acidente,0
municipio,0


In [9]:
df["classificacao_acidente"].value_counts()

,count
classificacao_acidente,
Com Vítimas Feridas,65066
Sem Vítimas,12867
Com Vítimas Fatais,5974


In [10]:
df["classificacao_acidente"].value_counts(normalize=True) * 100

,proportion
classificacao_acidente,
Com Vítimas Feridas,77.545378
Sem Vítimas,15.334835
Com Vítimas Fatais,7.119787


## 4. Tratamento de valores ausentes disfarçados

Além dos valores ausentes explícitos identificados com `isnull()`, a Entrega 1 mostrou que o dataset possui valores ausentes disfarçados, como `"Ignorado"` e `"Não Informado"`.

Esses valores representam ausência de informação, mas aparecem como texto. Por isso, antes da etapa de imputação, eles serão substituídos por `np.nan`.

In [11]:
df_limpo = df.copy()

valores_ausentes_disfarcados = [
    "Ignorado",
    "Não Informado",
    "Não informado",
    "Nao Informado",
    "Nao informado",
    "",
    " "
]

df_limpo = df_limpo.replace(valores_ausentes_disfarcados, np.nan)

df_limpo.isnull().sum().sort_values(ascending=False).head(15)

,0
condicao_metereologica,1114
sentido_via,192
uop,48
delegacia,27
regional,4
classificacao_acidente,2
br,0
uf,0
horario,0
km,0


In [12]:
print("Valores ausentes antes do tratamento:")
print(df.isnull().sum().sum())

print("\nValores ausentes depois do tratamento dos ausentes disfarçados:")
print(df_limpo.isnull().sum().sum())

Valores ausentes antes do tratamento:
81

Valores ausentes depois do tratamento dos ausentes disfarçados:
1387


Após a substituição dos valores ausentes disfarçados por `np.nan`, o número total de valores ausentes passou de 81 para 1387. Isso confirma o diagnóstico da Entrega 1: parte relevante da ausência de dados estava mascarada por categorias textuais como `"Ignorado"` e `"Não Informado"`.

As colunas mais impactadas foram `condicao_meteorologica` e `sentido_via`, que agora poderão ser tratadas corretamente nas etapas de imputação do pipeline.

## 5. Correção de inconsistências e remoção de registros problemáticos

A Entrega 1 identificou um registro anômalo, com quantidade incompatível de veículos e pessoas envolvidas. Como esse registro representa uma inconsistência evidente, ele será removido antes da modelagem.

Além disso, registros sem valor na variável-alvo `classificacao_acidente` também serão removidos, pois não podem ser utilizados no treinamento supervisionado do modelo kNN.

In [13]:
df_limpo = df_limpo.dropna(subset=["classificacao_acidente"])

condicao_anomala = (
    (df_limpo["veiculos"] > 50) &
    (df_limpo["pessoas"] <= 5)
)

print("registros anômalos encontrados:")
print(df_limpo[condicao_anomala][["id", "tipo_acidente", "pessoas", "veiculos", "ignorados"]])

df_limpo = df_limpo[~condicao_anomala]

print("dimensões após remoção dos registros problemáticos:")
print(df_limpo.shape)

registros anômalos encontrados:
           id tipo_acidente  pessoas  veiculos  ignorados
10260  707992      Incêndio        2        82         81
dimensões após remoção dos registros problemáticos:
(83906, 30)


A etapa confirmou a presença do registro anômalo identificado na Entrega 1: um acidente do tipo `Incêndio` com 82 veículos, apenas 2 pessoas e 81 ignorados. Por representar uma inconsistência evidente no preenchimento dos dados, esse registro foi removido.

Além disso, foram removidos os registros sem valor na variável-alvo `classificacao_acidente`, já que eles não poderiam ser utilizados no treinamento supervisionado do modelo kNN.

Após essa etapa, o dataset passou a ter 83.906 registros.

## 6. Seleção de variáveis para modelagem

Nesta etapa, foram removidas variáveis administrativas, identificadores e colunas que poderiam gerar vazamento de informação ou redundância na modelagem.

Como a variável-alvo `classificacao_acidente` representa a gravidade do acidente, colunas diretamente relacionadas ao resultado final, como `mortos`, `feridos_leves`, `feridos_graves`, `feridos`, `ilesos`, `ignorados` e `pessoas`, podem revelar a resposta ao modelo. Por isso, essas variáveis foram removidas nesta primeira versão do pipeline.

Também foram removidas colunas administrativas ou de alta cardinalidade sem uso direto nesta etapa inicial, como `id`, `municipio`, `regional`, `delegacia` e `uop`.

In [14]:
colunas_remover = [
    "id",
    "municipio",
    "regional",
    "delegacia",
    "uop",
    "mortos",
    "feridos_leves",
    "feridos_graves",
    "feridos",
    "ilesos",
    "ignorados",
    "pessoas"
]

colunas_remover = [coluna for coluna in colunas_remover if coluna in df_limpo.columns]

df_modelo = df_limpo.drop(columns=colunas_remover)

print("colunas removidas:")
print(colunas_remover)

print("\ndimensões após seleção de variáveis:")
print(df_modelo.shape)

df_modelo.head()

colunas removidas:
['id', 'municipio', 'regional', 'delegacia', 'uop', 'mortos', 'feridos_leves', 'feridos_graves', 'feridos', 'ilesos', 'ignorados', 'pessoas']

dimensões após seleção de variáveis:
(83906, 18)


,data_inversa,dia_semana,horario,uf,br,km,causa_acidente,tipo_acidente,classificacao_acidente,fase_dia,sentido_via,condicao_metereologica,tipo_pista,tracado_via,uso_solo,veiculos,latitude,longitude
0,2025-01-01,quarta-feira,06:20:00,SP,116,225,Reação tardia ou ineficiente do condutor,Tombamento,Com Vítimas Feridas,Pleno dia,Decrescente,Céu Claro,Múltipla,Reta;Declive,Sim,2,"-23,48586772","-46,54075317"
2,2025-01-01,quarta-feira,08:45:00,PR,369,"88,2",Reação tardia ou ineficiente do condutor,Colisão traseira,Com Vítimas Feridas,Pleno dia,Crescente,Sol,Dupla,Reta;Aclive,Sim,2,"-23,182565","-50,637228"
3,2025-01-01,quarta-feira,11:00:00,PR,116,74,Reação tardia ou ineficiente do condutor,Saída de leito carroçável,Com Vítimas Feridas,Pleno dia,Crescente,Céu Claro,Dupla,Reta,Não,2,"-25,36517687","-49,04223028"
4,2025-01-01,quarta-feira,09:30:00,MG,251,471,Velocidade Incompatível,Colisão frontal,Com Vítimas Feridas,Pleno dia,Decrescente,Chuva,Simples,Curva;Declive,Não,4,"-16,46801304","-43,43121303"
5,2025-01-01,quarta-feira,10:40:00,MT,70,669,Transitar na contramão,Colisão frontal,Com Vítimas Fatais,Pleno dia,Crescente,Céu Claro,Simples,Reta,Não,5,"-16,04148578","-57,25884017"


## 7. Engenharia de variáveis temporais

As colunas `data_inversa` e `horario` estavam em formato textual. Para aproveitá-las na modelagem, foram extraídas variáveis numéricas simples: o mês do acidente e a hora do dia.

Após essa extração, as colunas originais de data e horário foram removidas, pois o modelo kNN não trabalha diretamente com datas e horários em formato de texto.

In [15]:
df_modelo = df_modelo.copy()

df_modelo["data_inversa"] = pd.to_datetime(df_modelo["data_inversa"], errors="coerce")
df_modelo["mes"] = df_modelo["data_inversa"].dt.month

df_modelo["hora"] = pd.to_datetime(
    df_modelo["horario"],
    format="%H:%M:%S",
    errors="coerce"
).dt.hour

df_modelo = df_modelo.drop(columns=["data_inversa", "horario"])

print("dimensões após criação das variáveis temporais:")
print(df_modelo.shape)

df_modelo.head()

dimensões após criação das variáveis temporais:
(83906, 18)


,dia_semana,uf,br,km,causa_acidente,tipo_acidente,classificacao_acidente,fase_dia,sentido_via,condicao_metereologica,tipo_pista,tracado_via,uso_solo,veiculos,latitude,longitude,mes,hora
0,quarta-feira,SP,116,225,Reação tardia ou ineficiente do condutor,Tombamento,Com Vítimas Feridas,Pleno dia,Decrescente,Céu Claro,Múltipla,Reta;Declive,Sim,2,"-23,48586772","-46,54075317",1,6
2,quarta-feira,PR,369,"88,2",Reação tardia ou ineficiente do condutor,Colisão traseira,Com Vítimas Feridas,Pleno dia,Crescente,Sol,Dupla,Reta;Aclive,Sim,2,"-23,182565","-50,637228",1,8
3,quarta-feira,PR,116,74,Reação tardia ou ineficiente do condutor,Saída de leito carroçável,Com Vítimas Feridas,Pleno dia,Crescente,Céu Claro,Dupla,Reta,Não,2,"-25,36517687","-49,04223028",1,11
4,quarta-feira,MG,251,471,Velocidade Incompatível,Colisão frontal,Com Vítimas Feridas,Pleno dia,Decrescente,Chuva,Simples,Curva;Declive,Não,4,"-16,46801304","-43,43121303",1,9
5,quarta-feira,MT,70,669,Transitar na contramão,Colisão frontal,Com Vítimas Fatais,Pleno dia,Crescente,Céu Claro,Simples,Reta,Não,5,"-16,04148578","-57,25884017",1,10


## 8. Separação entre variáveis preditoras e variável-alvo

Nesta etapa, o dataset foi separado em variáveis preditoras (`X`) e variável-alvo (`y`). A variável-alvo escolhida foi `classificacao_acidente`, que representa a gravidade do acidente.

In [19]:
target = "classificacao_acidente"

X = df_modelo.drop(columns=[target])
y = df_modelo[target]

print("dimensões de X:", X.shape)
print("dimensões de y:", y.shape)

print("\ndistribuição da variável-alvo:")
print(y.value_counts(normalize=True) * 100)

dimensões de X: (83906, 17)
dimensões de y: (83906,)

distribuição da variável-alvo:
classificacao_acidente
Com Vítimas Feridas    77.546302
Sem Vítimas            15.333826
Com Vítimas Fatais      7.119872
Name: proportion, dtype: float64


## 9. Identificação de variáveis numéricas e categóricas

Para montar o pipeline de pré-processamento, as colunas foram separadas entre numéricas e categóricas. As variáveis numéricas serão imputadas e padronizadas, enquanto as categóricas serão imputadas e transformadas por encoding.

In [17]:
colunas_numericas = X.select_dtypes(include=["int64", "float64"]).columns
colunas_categoricas = X.select_dtypes(include=["object", "category"]).columns

print("colunas numéricas:")
print(list(colunas_numericas))

print("\ncolunas categóricas:")
print(list(colunas_categoricas))

colunas numéricas:
['br', 'veiculos']

colunas categóricas:
['dia_semana', 'uf', 'km', 'causa_acidente', 'tipo_acidente', 'fase_dia', 'sentido_via', 'condicao_metereologica', 'tipo_pista', 'tracado_via', 'uso_solo', 'latitude', 'longitude']


## 10. Correção de tipos numéricos

Algumas colunas numéricas foram carregadas como texto devido ao uso de vírgula como separador decimal. Para corrigir isso, as colunas `km`, `latitude` e `longitude` foram convertidas para formato numérico, substituindo vírgulas por pontos.

In [18]:
colunas_para_converter = ["km", "latitude", "longitude"]

for coluna in colunas_para_converter:
    df_modelo[coluna] = (
        df_modelo[coluna]
        .astype(str)
        .str.replace(",", ".", regex=False)
    )
    df_modelo[coluna] = pd.to_numeric(df_modelo[coluna], errors="coerce")

df_modelo[["km", "latitude", "longitude"]].dtypes

,0
km,float64
latitude,float64
longitude,float64


In [22]:
df_modelo = df_modelo.copy()

if "data_inversa" in df_modelo.columns:
    df_modelo["data_inversa"] = pd.to_datetime(df_modelo["data_inversa"], errors="coerce")
    df_modelo["mes"] = df_modelo["data_inversa"].dt.month

if "horario" in df_modelo.columns:
    df_modelo["hora"] = pd.to_datetime(
        df_modelo["horario"],
        format="%H:%M:%S",
        errors="coerce"
    ).dt.hour

colunas_temporais_remover = ["data_inversa", "horario"]
colunas_temporais_remover = [coluna for coluna in colunas_temporais_remover if coluna in df_modelo.columns]

df_modelo = df_modelo.drop(columns=colunas_temporais_remover)

df_modelo.head()

,dia_semana,uf,br,km,causa_acidente,tipo_acidente,classificacao_acidente,fase_dia,sentido_via,condicao_metereologica,tipo_pista,tracado_via,uso_solo,veiculos,latitude,longitude,mes,hora
0,quarta-feira,SP,116,225.0,Reação tardia ou ineficiente do condutor,Tombamento,Com Vítimas Feridas,Pleno dia,Decrescente,Céu Claro,Múltipla,Reta;Declive,Sim,2,-23.485868,-46.540753,1,6
2,quarta-feira,PR,369,88.2,Reação tardia ou ineficiente do condutor,Colisão traseira,Com Vítimas Feridas,Pleno dia,Crescente,Sol,Dupla,Reta;Aclive,Sim,2,-23.182565,-50.637228,1,8
3,quarta-feira,PR,116,74.0,Reação tardia ou ineficiente do condutor,Saída de leito carroçável,Com Vítimas Feridas,Pleno dia,Crescente,Céu Claro,Dupla,Reta,Não,2,-25.365177,-49.042230,1,11
4,quarta-feira,MG,251,471.0,Velocidade Incompatível,Colisão frontal,Com Vítimas Feridas,Pleno dia,Decrescente,Chuva,Simples,Curva;Declive,Não,4,-16.468013,-43.431213,1,9
5,quarta-feira,MT,70,669.0,Transitar na contramão,Colisão frontal,Com Vítimas Fatais,Pleno dia,Crescente,Céu Claro,Simples,Reta,Não,5,-16.041486,-57.258840,1,10


In [23]:
target = "classificacao_acidente"

X = df_modelo.drop(columns=[target])
y = df_modelo[target]

print("dimensões de X:", X.shape)
print("dimensões de y:", y.shape)

dimensões de X: (83906, 17)
dimensões de y: (83906,)


In [25]:
colunas_numericas = X.select_dtypes(include=[np.number]).columns
colunas_categoricas = X.select_dtypes(include=["object", "category"]).columns

print("colunas numéricas:")
print(list(colunas_numericas))

print("\ncolunas categóricas:")
print(list(colunas_categoricas))

colunas numéricas:
['br', 'km', 'veiculos', 'latitude', 'longitude', 'mes', 'hora']

colunas categóricas:
['dia_semana', 'uf', 'causa_acidente', 'tipo_acidente', 'fase_dia', 'sentido_via', 'condicao_metereologica', 'tipo_pista', 'tracado_via', 'uso_solo']


## 11. Pipeline de pré-processamento e modelo kNN

Nesta etapa, foi criado um pipeline para aplicar o pré-processamento de forma organizada.

Para as variáveis numéricas, foi utilizada imputação pela mediana e padronização com `StandardScaler`. A padronização é importante porque o kNN calcula distâncias entre os registros, então variáveis em escalas maiores poderiam dominar o resultado.

Para as variáveis categóricas, foi utilizada imputação pela moda e `OneHotEncoder`, transformando categorias textuais em variáveis numéricas binárias.

In [26]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

In [27]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("treino:", X_train.shape)
print("teste:", X_test.shape)

treino: (67124, 17)
teste: (16782, 17)


In [28]:
pipeline_numerico = Pipeline(steps=[
    ("imputador", SimpleImputer(strategy="median")),
    ("padronizador", StandardScaler())
])

pipeline_categorico = Pipeline(steps=[
    ("imputador", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessador = ColumnTransformer(transformers=[
    ("numerico", pipeline_numerico, colunas_numericas),
    ("categorico", pipeline_categorico, colunas_categoricas)
])

modelo_knn = Pipeline(steps=[
    ("preprocessamento", preprocessador),
    ("knn", KNeighborsClassifier(n_neighbors=5))
])

In [29]:
modelo_knn.fit(X_train, y_train)

y_pred = modelo_knn.predict(X_test)

In [30]:
print("acurácia:", accuracy_score(y_test, y_pred))

print("\nrelatório de classificação:")
print(classification_report(y_test, y_pred))

print("\nmatriz de confusão:")
print(confusion_matrix(y_test, y_pred))

acurácia: 0.7705279466094626

relatório de classificação:
                     precision    recall  f1-score   support

 Com Vítimas Fatais       0.26      0.12      0.17      1195
Com Vítimas Feridas       0.80      0.94      0.87     13014
        Sem Vítimas       0.55      0.19      0.28      2573

           accuracy                           0.77     16782
          macro avg       0.54      0.42      0.44     16782
       weighted avg       0.72      0.77      0.73     16782


matriz de confusão:
[[  147  1014    34]
 [  350 12294   370]
 [   61  2022   490]]


## 12. Avaliação inicial do kNN

O modelo kNN obteve acurácia de aproximadamente 77,05%. No entanto, essa métrica deve ser analisada com cuidado, pois a variável-alvo é desbalanceada: a classe `Com Vítimas Feridas` representa a maior parte dos registros.

O relatório de classificação mostra que o modelo teve melhor desempenho na classe majoritária, `Com Vítimas Feridas`, com recall de 0.94 e f1-score de 0.87. Por outro lado, o desempenho nas classes minoritárias foi menor, especialmente em `Com Vítimas Fatais`, que obteve recall de 0.12 e f1-score de 0.17.

Isso indica que, apesar da acurácia aparentemente alta, o modelo ainda tem dificuldade para identificar acidentes fatais e acidentes sem vítimas. Esse comportamento reforça o diagnóstico da Entrega 1 sobre o desbalanceamento da variável-alvo.

Como próximos passos, serão avaliadas estratégias para melhorar o desempenho nas classes minoritárias, como ajuste do valor de `k`, teste de diferentes formas de normalização e possíveis técnicas de balanceamento.